In [ ]:
# from pymongo import MongoClient

# client = MongoClient("mongodb://localhost:27017/")
# db = client["WhoScored"]

# collections = [
#     "available_teams",
#     "game_schedule",
#     "game_raw_events",
#     "game_processed_events",
#     "game_team_stats",
#     "game_player_stats",
#     "game_shot_sequences",
#     "game_pass_sequences",
#     "game_formations",
#     "game_lineups"
# ]

# for collection_name in collections:
#     collection = db[collection_name]
#     print(f"\n{collection_name}")

#     before = list(collection.list_indexes())
#     custom_indexes = [idx["name"] for idx in before if idx["name"] != "_id_"]

#     if not custom_indexes:
#         print("  no custom indexes to delete")
#         continue

#     for index_name in custom_indexes:
#         collection.drop_index(index_name)
#         print(f"  deleted: {index_name}")

# client.close()

In [ ]:
# from pymongo import MongoClient, ReplaceOne
# import os

# username = os.getenv("MONGO_USERNAME")
# password = os.getenv("MONGO_PASSWORD")
# host = os.getenv("MONGO_HOST")

# LOCAL_MONGO_URL = "mongodb://localhost:27017/"
# ATLAS_MONGO_URL = (
#     f"mongodb+srv://{username}:{password}@{host}/?retryWrites=true&w=majority&appName=Cluster0"
# )

# DB_NAME = "WhoScored"

# collections = [
#     "available_teams",
#     "game_schedule",
#     "game_raw_events",
#     "game_processed_events",
#     "game_team_stats",
#     "game_player_stats",
#     "game_shot_sequences",
#     "game_pass_sequences",
# ]

# local_client = MongoClient(LOCAL_MONGO_URL)
# atlas_client = MongoClient(ATLAS_MONGO_URL)

# local_db = local_client[DB_NAME]
# atlas_db = atlas_client[DB_NAME]

# for collection_name in collections:
#     print(f"\nCopying {collection_name}...")

#     local_collection = local_db[collection_name]
#     atlas_collection = atlas_db[collection_name]

#     total = local_collection.count_documents({})

#     if total == 0:
#         print("0 docs, skipped")
#         continue

#     batch = []
#     copied = 0

#     cursor = local_collection.find({'season': '2025-2026'}).batch_size(500)

#     for doc in cursor:
#         batch.append(ReplaceOne({"_id": doc["_id"]}, doc, upsert=True))

#         if len(batch) >= 500:
#             atlas_collection.bulk_write(batch, ordered=False)
#             copied += len(batch)
#             print(f"{collection_name}: copied {copied}/{total}")
#             batch = []

#     if batch:
#         atlas_collection.bulk_write(batch, ordered=False)
#         copied += len(batch)
#         print(f"{collection_name}: copied {copied}/{total}")

# print("\nAtlas databases:", atlas_client.list_database_names())
# print("Atlas WhoScored collections:", atlas_db.list_collection_names())

# local_client.close()
# atlas_client.close()

# print("\nDone.")

In [ ]:
# from pymongo import MongoClient, ASCENDING

# client = MongoClient("mongodb://localhost:27017/")
# db = client["WhoScored"]

# indexes = {
#     "available_teams": [
#         ([("ws_team_id", ASCENDING)], {
#             "unique": True,
#             "name": "uniq_ws_team_id",
#         }),
#         ([("ws_team_name", ASCENDING)], {
#             "name": "idx_ws_team_name",
#         }),
#         ([("fbref_team_name", ASCENDING)], {
#             "name": "idx_fbref_team_name",
#         }),
#     ],

#     "game_schedule": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {
#             "unique": True,
#             "name": "uniq_season_game",
#         }),
#         ([("season", ASCENDING), ("game_status", ASCENDING)], {
#             "name": "idx_season_status",
#         }),
#         ([("season", ASCENDING), ("game_date", ASCENDING)], {
#             "name": "idx_season_game_date",
#         }),
#         ([("season", ASCENDING), ("home_team_id", ASCENDING)], {
#             "name": "idx_season_home_team",
#         }),
#         ([("season", ASCENDING), ("away_team_id", ASCENDING)], {
#             "name": "idx_season_away_team",
#         }),
#     ],

#     "game_lineups": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {
#             "name": "idx_season_game",
#         }),
#         ([("season", ASCENDING), ("game_id", ASCENDING),
#           ("team_id", ASCENDING), ("player_id", ASCENDING)], {
#             "unique": True,
#             "name": "uniq_season_game_team_player",
#             "partialFilterExpression": {
#                 "team_id": {"$type": "number"},
#                 "player_id": {"$type": "number"},
#             },
#         }),
#         ([("season", ASCENDING), ("player_id", ASCENDING)], {
#             "name": "idx_season_player",
#             "partialFilterExpression": {
#                 "player_id": {"$type": "number"}
#             },
#         }),
#         ([("season", ASCENDING), ("game_id", ASCENDING),
#           ("starting_lineup", ASCENDING)], {
#             "name": "idx_season_game_starting_lineup",
#         }),
#     ],

#     "game_raw_events": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {
#             "name": "idx_season_game",
#         }),
#         ([("season", ASCENDING), ("game_id", ASCENDING),
#           ("event_idx", ASCENDING)], {
#             "unique": True,
#             "name": "uniq_season_game_event",
#         }),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {
#             "name": "idx_season_team",
#         }),
#         ([("season", ASCENDING), ("player_id", ASCENDING)], {
#             "name": "idx_season_player",
#             "partialFilterExpression": {
#                 "player_id": {"$type": "number"}
#             },
#         }),
#     ],

#     "game_processed_events": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {
#             "name": "idx_season_game",
#         }),
#         ([("season", ASCENDING), ("game_id", ASCENDING),
#           ("event_idx", ASCENDING)], {
#             "unique": True,
#             "name": "uniq_season_game_event",
#         }),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {
#             "name": "idx_season_team",
#         }),
#         ([("season", ASCENDING), ("player_id", ASCENDING)], {
#             "name": "idx_season_player",
#             "partialFilterExpression": {
#                 "player_id": {"$type": "number"}
#             },
#         }),
#         ([("season", ASCENDING), ("type", ASCENDING)], {
#             "name": "idx_season_type",
#         }),
#         ([("season", ASCENDING), ("qualifier_ids", ASCENDING)], {
#             "name": "idx_season_qualifier_ids",
#         }),
#     ],

#     "game_team_stats": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {
#             "name": "idx_season_game",
#         }),
#         ([("season", ASCENDING), ("game_id", ASCENDING),
#           ("team_id", ASCENDING)], {
#             "unique": True,
#             "name": "uniq_season_game_team",
#         }),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {
#             "name": "idx_season_team",
#         }),
#         ([("season", ASCENDING), ("opponent_team_id", ASCENDING)], {
#             "name": "idx_season_opponent",
#         }),
#     ],

#     "game_player_stats": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {
#             "name": "idx_season_game",
#         }),
#         ([("season", ASCENDING), ("game_id", ASCENDING),
#           ("player_id", ASCENDING)], {
#             "unique": True,
#             "name": "uniq_season_game_player",
#             "partialFilterExpression": {
#                 "player_id": {"$type": "number"}
#             },
#         }),
#         ([("season", ASCENDING), ("player_id", ASCENDING)], {
#             "name": "idx_season_player",
#             "partialFilterExpression": {
#                 "player_id": {"$type": "number"}
#             },
#         }),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {
#             "name": "idx_season_team",
#         }),
#     ],

#     "game_shot_sequences": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {
#             "name": "idx_season_game",
#         }),
#         ([("season", ASCENDING), ("game_id", ASCENDING),
#           ("sequence_key", ASCENDING), ("sequence_event", ASCENDING)], {
#             "unique": True,
#             "name": "uniq_shot_sequence_event",
#         }),
#         ([("season", ASCENDING), ("sequence_key", ASCENDING)], {
#             "name": "idx_season_sequence_key",
#         }),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {
#             "name": "idx_season_team",
#         }),
#     ],

#     "game_pass_sequences": [
#         ([("season", ASCENDING), ("game_id", ASCENDING)], {
#             "name": "idx_season_game",
#         }),
#         ([("season", ASCENDING), ("game_id", ASCENDING),
#           ("sequence_key", ASCENDING), ("sequence_event", ASCENDING)], {
#             "unique": True,
#             "name": "uniq_pass_sequence_event",
#         }),
#         ([("season", ASCENDING), ("sequence_key", ASCENDING)], {
#             "name": "idx_season_sequence_key",
#         }),
#         ([("season", ASCENDING), ("sequence_team_id", ASCENDING)], {
#             "name": "idx_season_sequence_team",
#         }),
#         ([("season", ASCENDING), ("team_id", ASCENDING)], {
#             "name": "idx_season_team",
#         }),
#     ],
# }

# for collection_name, specs in indexes.items():
#     collection = db[collection_name]
#     print(f"\n{collection_name}")

#     for keys, options in specs:
#         try:
#             index_name = collection.create_index(keys, **options)
#             print(f"  OK: {index_name}")
#         except Exception as exc:
#             print(f"  FAILED: {options['name']} -> {exc}")

# client.close()

In [ ]:
from pymongo import MongoClient
import pandas as pd
client = MongoClient("mongodb://localhost:27017/")
db = client["WhoScored"]

In [ ]:
collection_schedule = db["game_schedule"]
collection_raw_events = db["game_raw_events"]
collection_processing_events = db["game_processed_events"]
collection_team_stats = db["game_team_stats"]
collection_lineups = db["game_lineups"]
collection_formations = db["game_formations"]
collection_passing_sequence = db['game_pass_sequences']
collection_shot_sequence = db['game_shot_sequences']

In [ ]:
schedule_docs = list(collection_schedule.find(
    {},
    {"_id": 0, 'game_id': 1},
).sort("game_date", 1))

schedule_df = pd.DataFrame(collection_schedule.find({}))

In [ ]:
schedule_games = [doc.get('game_id') for doc in schedule_docs]


In [ ]:
game_formations = pd.DataFrame(collection_formations.find({}))
game_formations.groupby(['season'])['game_id'].value_counts().reset_index(drop=False)['count'].describe()

In [ ]:
game_formations.groupby(['season'])['game_id'].nunique()

In [ ]:
game_lineups = pd.DataFrame(collection_lineups.find({}))
game_lineups.groupby(['season'])['game_id'].value_counts().reset_index(drop=False)['count'].describe()

In [ ]:
game_lineups.groupby(['season'])['game_id'].nunique()

In [ ]:
valid_starting_lineups = game_lineups.groupby('game_id')['starting_lineup'].sum().reset_index(drop=False)
valid_starting_lineups['starting_lineup'].describe()

In [ ]:
def games_lineups_check(game):
    game_positions = game[game['starting_lineup']]['starting_position'].values.tolist()
    return "-".join(game_positions)

In [ ]:
check_positions = game_lineups.groupby(['game_id', 'team_name']).apply(games_lineups_check).reset_index(drop=False)

In [ ]:
check_positions = check_positions.rename(columns={0: 'positions'})
check_positions

In [ ]:
check_positions['positions'].value_counts(
)

In [ ]:
check_positions[check_positions['game_id'] == 328044]

In [ ]:
def infer_formation(position_string: str) -> str:
    positions = [position for position in position_string.split("-") if position and position != "GK"]

    defenders = sum(position in {"DR", "DC", "DL"} for position in positions)
    defensive_midfielders = sum(position in {"DMR", "DMC", "DML"} for position in positions)
    midfielders = sum(position in {"MR", "MC", "ML"} for position in positions)
    attacking_midfielders = sum(position in {"AMR", "AMC", "AML"} for position in positions)
    forwards = sum(position in {"FWR", "FW", "FWL"} for position in positions)

    if attacking_midfielders and forwards:
        middle = defensive_midfielders + midfielders
        return "-".join(
            str(number)
            for number in [defenders, middle, attacking_midfielders, forwards]
            if number > 0
        )

    midfield = defensive_midfielders + midfielders
    return "-".join(
        str(number)
        for number in [defenders, midfield, forwards]
        if number > 0
    )

In [ ]:
check_positions['formation'] = check_positions['positions'].apply(infer_formation)

In [ ]:
check_positions[check_positions['game_id'] == 328297]

In [ ]:
infer_formation('GK-DR-MC-DC-FW-MC-DC-ML-DL-MR-FW')

In [ ]:
game_formations['formation'].isnull().sum()

In [ ]:
game_formations['opponent_formation'].isnull().sum()

In [ ]:
game_formations['formation'].value_counts()

In [ ]:
game_formations['opponent_formation'].value_counts()

In [1]:
from pymongo import MongoClient
import pandas as pd
client = MongoClient("mongodb://localhost:27017/")
db = client["WhoScored"]

In [2]:
collection_schedule = db["game_schedule"]
# collection_raw_events = db["game_raw_events"]
# collection_processing_events = db["game_processed_events"]
# collection_team_stats = db["game_team_stats"]
# collection_lineups = db["game_lineups"]
# collection_formations = db["game_formations"]
# collection_passing_sequence = db['game_pass_sequences']
collection_shot_sequence = db['game_shot_sequences']

df_schedule = pd.DataFrame(collection_schedule.find({}, {'_id': 0, 'game_id': 1, 'season': 1}))
df_check = pd.DataFrame(collection_shot_sequence.find({}, {'_id': 0, 'game_id': 1, 'season': 1}))

In [3]:
df_check_game_ids = df_check['game_id'].unique().tolist()

In [4]:
df_schedule[~df_schedule['game_id'].isin(df_check_game_ids)]

,game_id,season
4039,1643097,2022-2023
4153,1643214,2022-2023
4777,1834391,2024-2025
